# 00 — Data: loading MNIST and the protocol splits

Everyone in the group loads the data the same way, through `nldr.data`.

- $X \in [0,1]^{70000 \times 784}$, one image per **row** (scikit-learn convention; the vault notes use columns, so `X` here is $X^\top$ there).
- Rows 0–59,999: official training set, the pool for all protocol splits. Rows 60,000–69,999: official test set, kept aside.
- Each repetition draws disjoint $X_{\mathrm{tr}}, X_{\mathrm{val}}, X_{\mathrm{te}}$ of size $n$ (`config.N_PER_SPLIT`). Same seed, same split, for everyone.

The first run downloads about 50 MB into `data/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make the nldr package importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nldr import config, data, metrics, methods, plotting

config.RESULTS_DIR.mkdir(exist_ok=True)
config.FIGURES_DIR.mkdir(exist_ok=True)

## 1. Load

In [ ]:
X, y = data.load_mnist()
print("X:", X.shape, X.dtype, "range", X.min(), X.max())
print("images per digit (training pool):", np.bincount(y[:60000]))

## 2. Protocol splits

In [ ]:
splits = data.make_splits()
s = splits[0]
print(f"{len(splits)} repetitions, n = {config.N_PER_SPLIT} per split")
print("split 0 sizes:", len(s.tr), len(s.val), len(s.te))
print("disjoint:", len(set(s.tr) | set(s.val) | set(s.te)) == 3 * config.N_PER_SPLIT)
print("digits in X_tr of split 0:", np.bincount(y[s.tr]))

## 3. Look at some digits

In [ ]:
fig = plotting.digit_images(X, s.tr[:30], titles=[str(d) for d in y[s.tr[:30]]])
plt.show()